# 🚀 15.7 骰子（APCS 2020-07 實作第 2 題）

本單元對應《Python 基礎與 APCS 檢定實戰》**第十五章：APCS 實作真題特訓（中級題）**。

> **🎯 適合對象**：國中進階資訊社團 / 高中職程式設計先修 / APCS 檢定衝刺學員  
> **🧭 學習路徑**：全課程微型單元 114 節之 **第 111 節**（實作真題系列）  
> **⚡ 核心概念**：六面骰三維空間旋轉模型、多重指派狀態輪替、1-based 陣列管理、對立面和為 7 幾何不變量檢驗。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/ipynb/PythAPCS123_15-7_dice_simulation_apcs_f580.ipynb)

---

## 📌 官方題目規格：APCS 實作真題 —— f580. 骰子

> 🏛️ **題目歷程與出處**  
> * **題目名稱**：骰子 (Dice Simulation)  
> * **檢定場次**：APCS 實作題檢定（2020 年 07 月實作題第 2 題）  
> * **線上評判**：ZeroJudge f580 / 高中生程式解題系統  
> * **推薦程度**：⭐⭐⭐⭐⭐（三維空間幾何旋轉與陣列狀態模擬之必考經典題）  

---

### 📖 題目敘述（Problem Description）
有 $n$ 顆傳統的六面立體骰子，由左至右排成一列，編號依序為 $1$ 到 $n$。  
傳統骰子的**相對兩面點數總和恆為 7**（即 $1$ 對 $6$、$2$ 對 $5$、$3$ 對 $4$）。

所有骰子一開始的擺放方向完全相同：
* **朝上（Top）** 的點數為 **1**（朝下 Bottom 為 6）
* **朝前（Front）** 的點數為 **4**（朝後 Back 為 3）
* **朝右（Right）** 的點數為 **2**（朝左 Left 為 5）

接下來會給定 $m$ 個操作指令，每個指令包含兩個整數 $a$ 與 $b$：
1. **位置對調操作（$a > 0$ 且 $b > 0$）**：  
   將編號 $a$ 的骰子與編號 $b$ 的骰子**整顆交換位置**（各自保持當前的旋轉姿態）。
2. **向前翻轉操作（$b = -1$）**：  
   將編號 $a$ 的骰子**向前翻轉 90 度**。
3. **向右翻轉操作（$b = -2$）**：  
   將編號 $a$ 的骰子**向右翻轉 90 度**。

請在依序執行完這 $m$ 次操作後，輸出這 $n$ 顆骰子最終**朝上（Top）**的點數。

---

### 📥 輸入格式（Input Format）
* 第一行包含兩個正整數 $n, m$：
  * $n$：骰子數量（$1 \le n \le 20$）
  * $m$：操作指令數量（$1 \le m \le 100$）
* 接下來 $m$ 行，每行包含兩個整數 $a, b$，代表一次操作指令：
  * 若 $a > 0$ 且 $b > 0$，代表交換編號 $a$ 與編號 $b$ 的骰子。
  * 若 $b = -1$，代表編號 $a$ 的骰子向前翻轉 90 度。
  * 若 $b = -2$，代表編號 $a$ 的骰子向右翻轉 90 度。

---

### 📤 輸出格式（Output Format）
依序輸出編號 $1$ 到編號 $n$ 骰子最終**朝上的點數**，數字之間以單一空格隔開，最後換行。

---

### 💡 官方範例解析

#### 範例一：
```text
1 2
1 -2
1 -1
```
* **初始狀態**：骰子 1 朝上 1、朝下 6、朝前 4、朝後 3、朝左 5、朝右 2。
* **指令 `1 -2`（向右翻轉 90 度）**：
  * 原本朝左的 5 翻轉到上面 $\to$ 朝上變 5。
  * 原本朝上的 1 翻轉到右面 $\to$ 朝右變 1。
  * 原本朝右的 2 翻轉到下面 $\to$ 朝下變 2。
  * 原本朝下的 6 翻轉到左面 $\to$ 朝左變 6。
  * 朝前（4）與朝後（3）保持不變。
* **指令 `1 -1`（向前翻轉 90 度）**：
  * 原本朝後的 3 翻轉到上面 $\to$ 朝上變 **3**。
  * 原本朝上的 5 翻轉到前面 $\to$ 朝前變 5。
* **最終朝上**：`3`。

#### 範例二：
```text
3 3
2 -1
3 -2
3 1
```
* **指令 1：`2 -1`** $\to$ 骰子 2 向前翻，朝上變 3。
* **指令 2：`3 -2`** $\to$ 骰子 3 向右翻，朝上變 5。
* **指令 3：`3 1`** $\to$ 交換骰子 3 與 骰子 1。
* **最終朝上點數**：骰子 1 為 5、骰子 2 為 3、骰子 3 為 1 $\to$ 輸出 `5 3 1`！

### 15.7.1 題意解析與空間幾何心智模型：多骰子並列與三維旋轉自由度

#### 💡 核心心智模型：
立體骰子在空間中有 3 個自由度，但本題只允許兩種特定旋轉方向：
1. **向前翻轉（Pitch Forward，指令 -1）**：繞左右軸轉動，影響「上、下、前、後」四個面，而「左、右」面完全不動。
2. **向右翻轉（Roll Right，指令 -2）**：繞前後軸轉動，影響「上、下、左、右」四個面，而「前、後」面完全不動。
3. **兩骰交換（Swap，指令 a, b > 0）**：整顆骰子的 6 面姿態完整互換。

```text
        [上 Top: 1]
             ▲
[左 Left: 5] ┼ [右 Right: 2]     (後 Back: 3 隱藏在背面)
             ▼
       [前 Front: 4]
       [下 Down: 6]
```


In [ ]:
# =====================================================================
# [2] Code 範例：骰子初始狀態與對面相加為 7 檢驗
# =====================================================================

# 標準骰子六面狀態表示法：[top, down, front, back, left, right]
init_dice = [1, 6, 4, 3, 5, 2]

top, down, front, back, left, right = init_dice
print("朝上面 (Top):", top)
print("朝前面 (Front):", front)
print("朝右面 (Right):", right)
print("對立面相加檢驗：", top + down == 7 and front + back == 7 and left + right == 7)


In [ ]:
# =====================================================================
# [3] Code 填空題：對面點數快速計算
# 任務：請將下方的 ___ 替換為 7 - top
# =====================================================================

current_top = 1
opposite_down = ___
print("朝下面點數：", opposite_down)  # 應為 6


In [ ]:
# =====================================================================
# [4] Code 練習題：建立 n 顆獨立初始骰子清單
# =====================================================================

def create_dice_list(n):
    # 使用深層列表，確保每顆骰子修改時互不干擾
    return [[1, 6, 4, 3, 5, 2] for _ in range(n + 1)]

d_list = create_dice_list(3)
assert len(d_list) == 4
assert d_list[1] is not d_list[2]
print("✅ 骰子陣列獨立初始化驗證通過！")


In [ ]:
# =====================================================================
# [5] Code 挑戰題：操作指令分類識別器
# =====================================================================

def identify_op(a, b):
    if b == -1:
        return f"骰子 {a} 向前旋轉"
    elif b == -2:
        return f"骰子 {a} 向右旋轉"
    else:
        return f"交換骰子 {a} 與 {b}"

assert identify_op(1, -1) == "骰子 1 向前旋轉"
assert identify_op(2, -2) == "骰子 2 向右旋轉"
assert identify_op(1, 3) == "交換骰子 1 與 3"
print("指令識別器檢驗全數合格！")


### 15.7.2 骰子六面體資料結構設計：標準展開圖與 6 面命名定位（上1下6、前4後3、左5右2）

#### 🎲 資料結構選型指南：
在競賽中，儲存一顆骰子的狀態有三種流派：
1. **六元素串列流派（最推薦）**：`dice = [top, down, front, back, left, right]`。
   * 優點：命名透明直觀，旋轉置換時一對一映射，絕不混淆。
2. **字典結構流派**：`{'top': 1, 'down': 6, ...}`。
   * 優點：可讀性高；缺點：存取速度稍慢。
3. **三維極限流派**：只存 `[top, front, right]`，其餘由 7 減去。
   * 優點：記憶體極小；缺點：心算置換公式容易考場翻車。

為了讓新手與考場高壓下的同學**穩健奪分**，我們全面採用**六元素串列**結構！


In [ ]:
# =====================================================================
# [2] Code 範例：骰子類別與六面封裝示範
# =====================================================================

class Dice:
    def __init__(self):
        self.top = 1
        self.down = 6
        self.front = 4
        self.back = 3
        self.left = 5
        self.right = 2
        
    def __repr__(self):
        return f"Dice(上={self.top}, 下={self.down}, 前={self.front}, 後={self.back}, 左={self.left}, 右={self.right})"

d1 = Dice()
print("骰子物件封裝狀態：", d1)


In [ ]:
# =====================================================================
# [3] Code 填空題：索引常數定義
# 任務：請將下方的 ___ 替換為 0 代表 top 的索引位置
# =====================================================================

IDX_TOP = ___
d_arr = [1, 6, 4, 3, 5, 2]
print("朝上點數提取：", d_arr[IDX_TOP])  # 應為 1


In [ ]:
# =====================================================================
# [4] Code 練習題：六面有效性約束檢驗函式
# =====================================================================

def is_valid_dice(d):
    # 1. 包含 1 到 6 的所有數字
    # 2. 三對對面總和皆為 7
    t, dw, f, b, l, r = d
    has_all_digits = (sorted(d) == [1, 2, 3, 4, 5, 6])
    sum_seven = (t + dw == 7 and f + b == 7 and l + r == 7)
    return has_all_digits and sum_seven

assert is_valid_dice([1, 6, 4, 3, 5, 2]) == True
assert is_valid_dice([1, 1, 4, 3, 5, 2]) == False
print("✅ 骰子六面完整性約束檢驗通過！")


In [ ]:
# =====================================================================
# [5] Code 挑戰題：深層複製品質保證
# =====================================================================

original_d = [1, 6, 4, 3, 5, 2]
cloned_d = original_d[:]
cloned_d[0] = 99
assert original_d[0] == 1
print("切片複本獨立性防禦合格！")


### 15.7.3 旋轉變換推導一：向前翻轉（指令 -1）之四面輪替置換公式

#### 🔄 向前翻轉（Forward Rotation）幾何推導：
想像你在桌上將一顆骰子**往前推滾動一格（90度）**：
* 原本在**背面（Back）**的點數轉到了**頂面（Top）**：$\text{new\_top} = \text{old\_back}$
* 原本在**頂面（Top）**的點數轉到了**正面（Front）**：$\text{new\_front} = \text{old\_top}$
* 原本在**正面（Front）**的點數轉到了**底面（Down）**：$\text{new\_down} = \text{old\_front}$
* 原本在**底面（Down）**的點數轉到了**背面（Back）**：$\text{new\_back} = \text{old\_down}$
* **左面（Left）與右面（Right）**：繞左右軸轉動，點數**完全不受影響**！

#### 🐍 Python 多重指派單行神技：
```python
# 向前翻轉：
top, down, front, back = back, front, top, down
```


In [ ]:
# =====================================================================
# [2] Code 範例：向前翻轉函式實作
# =====================================================================

def roll_forward(d):
    # d = [top, down, front, back, left, right]
    top, down, front, back, left, right = d
    # 多重平行指派，杜絕時序覆蓋問題
    d[0], d[1], d[2], d[3] = back, front, top, down

test_d = [1, 6, 4, 3, 5, 2]
roll_forward(test_d)
print("向前翻轉一次後狀態 [上, 下, 前, 後, 左, 右]：", test_d)
# 預期：原 back(3) 到 top(0)，原 top(1) 到 front(2) -> [3, 4, 1, 6, 5, 2]


In [ ]:
# =====================================================================
# [3] Code 填空題：向前翻轉之頂面來源
# 任務：請將下方的 ___ 替換為 back
# =====================================================================

top, down, front, back = 1, 6, 4, 3
new_top = ___
print("向前翻轉後朝上點數：", new_top)  # 應為 3


In [ ]:
# =====================================================================
# [4] Code 練習題：向前翻轉四次週期性（回原位）驗證
# =====================================================================

cycle_d = [1, 6, 4, 3, 5, 2]
for _ in range(4):
    roll_forward(cycle_d)

assert cycle_d == [1, 6, 4, 3, 5, 2]
print("✅ 向前翻轉四次回歸初始狀態驗證通過！")


In [ ]:
# =====================================================================
# [5] Code 挑戰題：向前翻轉後依然保持相對面和為 7 幾何驗證
# =====================================================================

assert is_valid_dice(test_d) == True
print("翻轉後相對面和為 7 恆成立驗證通過！")


### 15.7.4 旋轉變換推導二：向右翻轉（指令 -2）之四面輪替置換公式

#### 🔄 向右翻轉（Right Rotation）幾何推導：
想像你在桌上將骰子**往右側翻滾一格（90度）**：
* 原本在**左面（Left）**的點數轉到了**頂面（Top）**：$\text{new\_top} = \text{old\_left}$
* 原本在**頂面（Top）**的點數轉到了**右面（Right）**：$\text{new\_right} = \text{old\_top}$
* 原本在**右面（Right）**的點數轉到了**底面（Down）**：$\text{new\_down} = \text{old\_right}$
* 原本在**底面（Down）**的點數轉到了**左面（Left）**：$\text{new\_left} = \text{old\_down}$
* **正面（Front）與背面（Back）**：繞前後軸轉動，點數**完全不受影響**！

#### 🐍 Python 多重指派單行神技：
```python
# 向右翻轉：
top, down, left, right = left, right, down, top
```


In [ ]:
# =====================================================================
# [2] Code 範例：向右翻轉函式實作
# =====================================================================

def roll_right(d):
    # d = [top, down, front, back, left, right]
    top, down, front, back, left, right = d
    # 多重平行指派，更新 top, down, left, right 四面
    d[0], d[1], d[4], d[5] = left, right, down, top

test_r = [1, 6, 4, 3, 5, 2]
roll_right(test_r)
print("向右翻轉一次後狀態：", test_r)
# 預期：原 left(5) 到 top(0)，原 top(1) 到 right(5) -> [5, 2, 4, 3, 6, 1]


In [ ]:
# =====================================================================
# [3] Code 填空題：向右翻轉之頂面來源
# 任務：請將下方的 ___ 替換為 left
# =====================================================================

top, down, left, right = 1, 6, 5, 2
new_top_right = ___
print("向右翻轉後朝上點數：", new_top_right)  # 應為 5


In [ ]:
# =====================================================================
# [4] Code 練習題：向右翻轉四次週期性（回原位）驗證
# =====================================================================

cycle_r = [1, 6, 4, 3, 5, 2]
for _ in range(4):
    roll_right(cycle_r)

assert cycle_r == [1, 6, 4, 3, 5, 2]
print("✅ 向右翻轉四次週期性驗證通過！")


In [ ]:
# =====================================================================
# [5] Code 挑戰題：官方範例一步驟還原推演
# =====================================================================

d_sample1 = [1, 6, 4, 3, 5, 2]
roll_right(d_sample1)   # 指令 1 -2
roll_forward(d_sample1) # 指令 1 -1
assert d_sample1[0] == 3
print("官方範例一 (1 -2 後 1 -1) 答案 3 驗證成功！")


### 15.7.5 骰子陣列管理與位置對調操作：指令大於 0 時的兩骰狀態交換（Swap）

#### 🔄 兩骰位置對調（Swap Operation）：
當指令中的兩個數值 $a > 0$ 且 $b > 0$ 時：
* 題目要求將「編號 $a$ 的骰子」與「編號 $b$ 的骰子」**位置對調**！
* 這代表兩顆骰子完整互換身分與位置，各自的姿態點數完全保留。

在 Python 中，兩物件對調僅需一行俐落代碼：
```python
dice[a], dice[b] = dice[b], dice[a]
```


In [ ]:
# =====================================================================
# [2] Code 範例：兩骰交換實作示範
# =====================================================================

# 假設有 3 顆骰子（1-based 陣列）
dice_array = [
    None,
    [1, 6, 4, 3, 5, 2],  # 骰子 1 朝上 1
    [3, 4, 1, 6, 5, 2],  # 骰子 2 朝上 3
    [5, 2, 4, 3, 6, 1]   # 骰子 3 朝上 5
]

# 交換骰子 1 與 骰子 3
dice_array[1], dice_array[3] = dice_array[3], dice_array[1]

print("交換後骰子 1 朝上：", dice_array[1][0])  # 5
print("交換後骰子 3 朝上：", dice_array[3][0])  # 1


In [ ]:
# =====================================================================
# [3] Code 填空題：Python 雙向對調語法
# 任務：請將下方的 ___ 替換為 arr[b], arr[a]
# =====================================================================

arr = ["None", "Dice_A", "Dice_B"]
a, b = 1, 2
arr[a], arr[b] = ___
print("對調後第一個元素：", arr[1])  # 應為 Dice_B


In [ ]:
# =====================================================================
# [4] Code 練習題：同一顆骰子自我對調（a == b）邊界防禦
# =====================================================================

self_swap_d = [None, [1, 6, 4, 3, 5, 2]]
self_swap_d[1], self_swap_d[1] = self_swap_d[1], self_swap_d[1]
assert self_swap_d[1] == [1, 6, 4, 3, 5, 2]
print("✅ 自我對調安全防禦測試通過！")


In [ ]:
# =====================================================================
# [5] Code 挑戰題：多次連續交換之排列一致性驗證
# =====================================================================

order = [None, "A", "B", "C"]
# 交換 1, 2 ➔ [None, 'B', 'A', 'C']
order[1], order[2] = order[2], order[1]
# 交換 2, 3 ➔ [None, 'B', 'C', 'A']
order[2], order[3] = order[3], order[2]
assert order[1:] == ["B", "C", "A"]
print("連續交換置換檢驗通過！")


### 15.7.6 1-based 索引對齊與多指令串流讀取主迴圈建構

#### 🎯 1-based 索引對齊防呆技巧：
題目的骰子編號是 $1 \sim n$。如果我們宣告長度為 $n+1$ 的串列：
`dice = [None] + [[1, 6, 4, 3, 5, 2] for _ in range(n)]`
* 骰子 $1$ 直接對應 `dice[1]`，骰子 $n$ 直接對應 `dice[n]`！
* 完全不需要在讀取指令後手動減 1（`- 1`），**徹底杜絕考場差 1 錯誤（Off-by-One Error）**！

#### 🔄 主控制迴圈架構：
```python
for _ in range(m):
    a, b = map(int, input().split())
    if b > 0:
        dice[a], dice[b] = dice[b], dice[a]
    elif b == -1:
        roll_forward(dice[a])
    elif b == -2:
        roll_right(dice[a])
```


In [ ]:
# =====================================================================
# [2] Code 範例：主控制迴圈模擬器原型
# =====================================================================

def simulate_dice_ops(n, m, ops):
    # 1-based 索引清單
    dice = [None] + [[1, 6, 4, 3, 5, 2] for _ in range(n)]
    
    for a, b in ops:
        if b > 0:
            dice[a], dice[b] = dice[b], dice[a]
        elif b == -1:
            roll_forward(dice[a])
        elif b == -2:
            roll_right(dice[a])
            
    # 收集朝上點數
    return [dice[i][0] for i in range(1, n + 1)]

sample2_ops = [[2, -1], [3, -2], [3, 1]]
result_2 = simulate_dice_ops(3, 3, sample2_ops)
print("官方範例二模擬結果：", result_2)  # [5, 3, 1]


In [ ]:
# =====================================================================
# [3] Code 填空題：收集各骰子朝上頂面
# 任務：請將下方的 ___ 替換為 dice[i][0]
# =====================================================================

mock_dice = [None, [5, 2, 4, 3, 6, 1], [3, 4, 1, 6, 5, 2]]
top_faces = [___ for i in range(1, 3)]
print("各骰子朝上列表：", top_faces)  # 應為 [5, 3]


In [ ]:
# =====================================================================
# [4] Code 練習題：驗證官方範例一與範例二
# =====================================================================

assert simulate_dice_ops(1, 2, [[1, -2], [1, -1]]) == [3]
assert simulate_dice_ops(3, 3, [[2, -1], [3, -2], [3, 1]]) == [5, 3, 1]
print("✅ 官方範例一與範例二全數精準通過！")


In [ ]:
# =====================================================================
# [5] Code 挑戰題：零操作（m=0）邊界狀態檢驗
# =====================================================================

init_tops = simulate_dice_ops(4, 0, [])
assert init_tops == [1, 1, 1, 1]
print("m=0 零操作邊界檢驗安全通過！")


### 15.7.7 完整 AC 模組拼裝：各骰子頂面點數依序解包輸出

#### 📤 競技輸出規範：
題目要求將最終 $n$ 顆骰子朝上的點數**以單一空格隔開輸出**：
* 推薦寫法：`print(*(dice[i][0] for i in range(1, n + 1)))` 或 `print(*top_faces)`。
* 星號解包（`*`）會自動在元素間填入空白，並在尾端加上換行，完美符合 APCS 與 ZeroJudge 評判規範！


In [ ]:
# =====================================================================
# [2] Code 範例：解包輸出展示
# =====================================================================

demo_ans = [5, 3, 1]
print("解包輸出效果：")
print(*demo_ans)


In [ ]:
# =====================================================================
# [3] Code 填空題：星號解包輸出
# 任務：請將下方的 ___ 替換為 *ans_list
# =====================================================================

ans_list = [1, 2, 3, 4]
import io, sys
buf = io.StringIO()
print(___, file=buf)
print("輸出字串：", repr(buf.getvalue()))  # 應為 '1 2 3 4\n'


In [ ]:
# =====================================================================
# [4] Code 練習題：格式化單一數值解包
# =====================================================================

single_ans = [3]
buf_single = io.StringIO()
print(*single_ans, file=buf_single)
assert buf_single.getvalue() == "3\n"
print("✅ 單一元素格式化解包通過！")


In [ ]:
# =====================================================================
# [5] Code 挑戰題：大量點數單行格式化輸出檢驗
# =====================================================================

twenty_dice = [1] * 20
buf_20 = io.StringIO()
print(*twenty_dice, file=buf_20)
assert len(buf_20.getvalue().split()) == 20
print("20 顆骰子極限輸出格式檢驗無誤！")


### 15.7.8 對立面相加恆為 7 的自檢驗證技巧、空間複雜度分析與考場失分陷阱排查

#### ⏱️ 複雜度分析：
* **時間複雜度**：
  * 骰子數 $n \le 20$，操作指令數 $m \le 100$。
  * 每次旋轉或交換均為 $O(1)$ 基本賦值操作。
  * 總時間複雜度為 $O(m) \le 100$ 次操作，在 Python 中執行耗時**小於 0.001 秒**！
* **空間複雜度**：
  * 僅維護 $n+1$ 個長度為 6 的整數陣列，空間為 $O(n) \le 120$ 個整數，記憶體佔用小於幾 KB，絕對安全！

#### 💣 考場 3 大失分地雷自我檢核表：
1. **向前與向右輪替搞反**：  
   * 向前（-1）：繞左右軸（前、後、上、下變動，左、右不動）。
   * 向右（-2）：繞前後軸（左、右、上、下變動，前、後不動）。
2. **輪替方向倒退（前進變倒退）**：  
   請務必手動在紙上畫出展開圖推導一次，確認 `top` 到底接收誰的值！
3. **0-based 與 1-based 混淆**：  
   題目給予的編號為 $1 \sim n$，陣列宣告長度必須為 $n+1$，直接對齊，切勿手忙腳亂亂減 1！


In [ ]:
# =====================================================================
# [2] Code 範例：極限隨機 100 次操作壓力測試與不變量自檢器
# =====================================================================

import random

random.seed(42)
sim_n = 20
sim_dice = [None] + [[1, 6, 4, 3, 5, 2] for _ in range(sim_n)]

# 隨機產生 100 次合法操作
for step in range(100):
    op_type = random.choice([-1, -2, 1])
    a = random.randint(1, sim_n)
    if op_type in (-1, -2):
        if op_type == -1: roll_forward(sim_dice[a])
        else: roll_right(sim_dice[a])
    else:
        b = random.randint(1, sim_n)
        sim_dice[a], sim_dice[b] = sim_dice[b], sim_dice[a]
        
# 驗證每顆骰子對立面總和是否依然恆為 7
for i in range(1, sim_n + 1):
    assert is_valid_dice(sim_dice[i]), f"骰子 {i} 幾何約束破損！"

print("100 次隨機複合操作壓力測試通過！所有骰子對立面和恆為 7 完全守恆！")


In [ ]:
# =====================================================================
# [3] Code 填空題：最大操作次數估算
# 任務：請將下方的 ___ 替換為 100
# =====================================================================

max_m_ops = ___
print("最大操作次數：", max_m_ops)  # 應為 100


In [ ]:
# =====================================================================
# [4] Code 練習題：多組邊界自動對拍套件
# =====================================================================

test_cases = [
    {
        "name": "官方範例一 (1 骰子, 2 旋轉)",
        "n": 1, "m": 2,
        "ops": [[1, -2], [1, -1]],
        "expected": [3]
    },
    {
        "name": "官方範例二 (3 骰子, 旋轉加對調)",
        "n": 3, "m": 3,
        "ops": [[2, -1], [3, -2], [3, 1]],
        "expected": [5, 3, 1]
    },
    {
        "name": "極端單顆連續四次向右翻轉回原位",
        "n": 1, "m": 4,
        "ops": [[1, -2], [1, -2], [1, -2], [1, -2]],
        "expected": [1]
    }
]

for tc in test_cases:
    res = simulate_dice_ops(tc["n"], tc["m"], tc["ops"])
    assert res == tc["expected"], f"{tc['name']} 失敗！預期 {tc['expected']}, 得到 {res}"
    print(f"✅ PASS: {tc['name']} ➔ 朝上結果: {res}")


In [ ]:
# =====================================================================
# [5] Code 挑戰題：極速單行解推導對拍
# =====================================================================

# 驗證三維精簡態 [top, front, right] 是否與六面完全等價
t, f, r = 1, 4, 2
# 向右：new_t = 7 - r, new_f = f, new_r = t (原 left=7-r 變 top，原 top 變 right)
t, r = 7 - r, t  # 5, 1
# 向前：new_t = 7 - f, new_f = t, new_r = r (原 back=7-f 變 top，原 top 變 front)
t, f = 7 - f, t  # 3, 5
assert t == 3
print("三維極簡態公式對拍成功！")


---

## 🏆 恭喜通關！單元 15.7 學習總結、今日解鎖能力盤點與榮耀通關徽章

### 🌟 今日解鎖的核心解題超能力
恭喜各位程式冒險者成功拿下 **APCS 實作中級題：f580. 骰子**！  
在攻克這道經典空間幾何模擬題的 8 大微階梯特訓中，大家已經全面升級並掌握了以下關鍵能力：
1. **空間幾何自由度心智模型**：精確區分向前旋轉（繞左右軸）與向右旋轉（繞前後軸）的四面置換機制。
2. **六面骰資料結構設計**：掌握 `[top, down, front, back, left, right]` 六維清單的透明可讀性與維護方式。
3. **Python 多重平行指派**：以一行代碼完成四面輪替置換，徹底杜絕時序變數覆蓋問題。
4. **1-based 陣列索引對齊**：掌握前綴 `[None]` 的防呆陣列管理，考場零失誤免除 `-1` 困擾。
5. **數學不變量自檢法**：學會利用「對立面和恆為 7」進行演算法自我驗證，高壓檢定下 100% 奪得滿分！

```text
┌────────────────────────────────────────────────────────────────────────┐
│                     🏆 APCS 實作真題特訓通關勳章                       │
├────────────────────────────────────────────────────────────────────────┤
│  恭喜完成 APCS 實作真題中級題：f580. 骰子（50 Cells 完整版）            │
│                                                                        │
│  解鎖技能：三維空間旋轉推導、四面輪替多重指派、兩骰狀態置換、幾何守恆    │
│  成就認證：100% 通過官方範例測資與 100 次隨機複合操作壓力測試！          │
└────────────────────────────────────────────────────────────────────────┘
```


---

## 💻 【附錄：雙平台滿分通關解答庫】考 APCS vs 刷 ZeroJudge 之差異與標準寫法

在 APCS 程式設計實作訓練中，初學冒險者最常遇到一個極具代表性的疑惑：
> **「為什麼同一題程式碼，在 APCS 考場可以拿到滿分，但在 ZeroJudge 卻可能會拿到 WA 或 NA？或者反過來，為什麼網路上看到的解題程式碼寫得那麼複雜？」**

這是因為 **APCS 官方考場** 與 **ZeroJudge 等線上解題系統（Online Judge, OJ）** 在評測資料的灌入機制上有根本性的不同，且每位同學在不同學習階段需要的代碼精簡度也不同。為此，我們特別提供**三種層次分明的滿分版本**：

| 評測與程式版本 | 適用情境與受眾推薦 | 核心寫法特色與優勢 |
| :--- | :--- | :--- |
| **🥇 版本一：APCS 淺顯易懂一般版** | APCS 正式考場（**初學者與一般程度同學首選**） | 步驟平鋪直敘、明確宣告六面陣列、清晰定義向前與向右滾動函式。邏輯透明清晰，考場高壓下最不易緊張出錯，穩健 100% AC！ |
| **⚡ 版本二：APCS 極簡高效精煉版** | APCS 正式考場（**進階同學與競賽衝刺者推薦**） | 僅維護上前右三面 `[top, front, right]`，利用對立面和為 7 的代數特性極速推導，僅約 16 行展現 Pythonic 高階極簡魅力！ |
| **🌐 版本三：ZeroJudge 萬用 AC 版** | ZeroJudge f580 / 高中生線上解題系統 | 支援連續多測資（EOF 串流解析）、模組化函式封裝、內建 Colab 本地全自動化測試套件，隨點隨測！ |

下方我們將三者分別提供為**完全獨立的程式碼區域**，供同學依據不同練習情境深入對照學習！


---

### 📝 版本一：APCS 官方實作考場專用版 —— 淺顯易懂一般版（新手友善推薦）

* **適用情境**：APCS 正式考試現場、初學者課堂練習。
* **教學與設計理念**：
  1. **步驟平鋪直敘、拆解詳細**：依序完成「初始化 $n$ 顆骰子六面狀態」、「定義向前滾動函式」、「定義向右滾動函式」、「解析指令執行」、「星號解包輸出」五大步驟。
  2. **1-based 索引無縫對齊**：陣列大小設為 $n+1$，完全不需減 1，零心算負擔。
  3. **穩健 100% 滿分 AC**：語意一目瞭然，考場高壓下最容易檢查除錯，零失誤奪取滿分！


In [ ]:
# ==============================================================================
# 📝 版本一：APCS 官方實作考場專用版 —— 淺顯易懂一般版（新手友善推薦）
# 適用情境：APCS 正式考場單筆測試案例，平鋪直敘、步驟明確、穩健滿分
# ==============================================================================

# 步驟 1：讀取骰子數量 n 與操作指令數量 m
n, m = map(int, input().split())

# 步驟 2：初始化 n 顆骰子（使用 1-based 索引，前方補 None）
# 每顆骰子狀態：[top, down, front, back, left, right]
dice = [None] + [[1, 6, 4, 3, 5, 2] for _ in range(n)]

# 定義向前翻轉 90 度函式（指令 -1）
def roll_forward(d):
    top, down, front, back, left, right = d
    # 原 back 轉到上面，原 top 轉到前面，原 front 轉到下面，原 down 轉到後面
    d[0], d[1], d[2], d[3] = back, front, top, down

# 定義向右翻轉 90 度函式（指令 -2）
def roll_right(d):
    top, down, front, back, left, right = d
    # 原 left 轉到上面，原 top 轉到右面，原 right 轉到下面，原 down 轉到左面
    d[0], d[1], d[4], d[5] = left, right, down, top

# 步驟 3：依序讀取並執行 m 個操作指令
for _ in range(m):
    a, b = map(int, input().split())
    # 情況一：位置對調操作 (a > 0 且 b > 0)
    if b > 0:
        dice[a], dice[b] = dice[b], dice[a]
    # 情況二：向前翻轉 (b == -1)
    elif b == -1:
        roll_forward(dice[a])
    # 情況三：向右翻轉 (b == -2)
    elif b == -2:
        roll_right(dice[a])

# 步驟 4：提取每顆骰子朝上的點數
ans = [dice[i][0] for i in range(1, n + 1)]

# 步驟 5：以空格隔開輸出
print(*ans)


---

### ⚡ 版本二：APCS 官方實作考場專用版 —— 極簡高效精煉版（進階高手推薦）

* **適用情境**：APCS 現場正式檢定考試（追求極速敲碼、極致優雅的高階模式）。
* **教學與設計理念**：
  1. **三面精簡態降維**：只維護 `[top, front, right]`，背面即為 `7 - front`，左面即為 `7 - right`。
  2. **簡約緊湊的 Pythonic 風格**：向右只需 `top, right = 7 - right, top`；向前只需 `top, front = 7 - front, top`。
  3. **約 16 行極速奪分**：程式碼高度精煉，展現純粹的代數之美！


In [ ]:
# ==============================================================================
# ⚡ 版本二：APCS 官方實作考場專用版 —— 極簡高效精煉版（進階高手推薦）
# 適用情境：APCS 正式考試現場，極簡 16 行代數降維模擬
# ==============================================================================

n, m = map(int, input().split())
# 僅維護 [top, front, right] 三面
d = [[1, 4, 2] for _ in range(n + 1)]

for _ in range(m):
    a, b = map(int, input().split())
    if b > 0:
        d[a], d[b] = d[b], d[a]
    elif b == -1:  # 向前：new_top = 7 - old_front, new_front = old_top
        d[a][0], d[a][1] = 7 - d[a][1], d[a][0]
    elif b == -2:  # 向右：new_top = 7 - old_right, new_right = old_top
        d[a][0], d[a][2] = 7 - d[a][2], d[a][0]

print(*(d[i][0] for i in range(1, n + 1)))


---

### 🌐 版本三：ZeroJudge 線上評判萬用 AC 版（支援多筆測資 EOF 循環）

* **適用情境**：ZeroJudge 線上刷題（題目代碼：f580）、各高中 OJ 競賽系統。
* **解題特點**：
  1. **處理連續多組測資機制**：  
     ZeroJudge 等線上 OJ 系統中，評測機可能將多組骰子模擬案例連續灌入。本版本使用 `sys.stdin.read().split()` 將所有數值一次讀入，使用指標循序解析，徹底杜絕 `EOFError`！
  2. **ZeroJudge 複製提交專區**：  
     下方特別劃分了【ZeroJudge 複製提交專區】，同學們只要複製該段程式碼，貼到 ZeroJudge f580 即可直接收穫 100% 滿分 AC！
  3. **Colab 本地自動化測試**：  
     下方儲存格已內建官方範例測資與極限邊界測資全自動化測試套件，點擊執行即可立即檢驗輸出結果！


In [ ]:
# ==============================================================================
# 🌐 版本三：ZeroJudge 線上評判萬用 AC 版（支援多筆測資 EOF 循環）
# ==============================================================================
import sys

def solve_dice_simulation(n, m, ops):
    """
    骰子旋轉與交換模擬核心演算法函式
    :param n: 骰子數量
    :param m: 指令數量
    :param ops: 指令清單，每項為 [a, b]
    :return: 各骰子頂面點數清單
    """
    # 1-based 索引初始化骰子列表：[top, down, front, back, left, right]
    d = [None] + [[1, 6, 4, 3, 5, 2] for _ in range(n)]
    
    for a, b in ops:
        if b > 0:
            d[a], d[b] = d[b], d[a]
        elif b == -1:  # 向前翻轉
            top, down, front, back, left, right = d[a]
            d[a][0], d[a][1], d[a][2], d[a][3] = back, front, top, down
        elif b == -2:  # 向右翻轉
            top, down, front, back, left, right = d[a]
            d[a][0], d[a][1], d[a][4], d[a][5] = left, right, down, top
            
    return [d[i][0] for i in range(1, n + 1)]

# ---------------------------------------------------------------------
# 【ZeroJudge 官方提交程式碼範本】
# 若要在 ZeroJudge 提交，請複製下方函式內容至解題系統：
# def main():
#     tokens = sys.stdin.read().split()
#     if not tokens: return
#     idx = 0
#     while idx < len(tokens):
#         n = int(tokens[idx])
#         m = int(tokens[idx+1])
#         idx += 2
#         ops = []
#         for _ in range(m):
#             ops.append([int(tokens[idx]), int(tokens[idx+1])])
#             idx += 2
#         ans = solve_dice_simulation(n, m, ops)
#         print(*ans)
# ---------------------------------------------------------------------

# =====================================================================
# 🧪 Colab 本地自動化測試檢驗套件（點擊播放鍵自動執行）
# =====================================================================
test_cases = [
    {
        "name": "官方範例一 (1 顆骰子, 先右轉後前轉)",
        "n": 1, "m": 2,
        "ops": [[1, -2], [1, -1]],
        "expected": [3]
    },
    {
        "name": "官方範例二 (3 顆骰子, 前轉、右轉與位置對調)",
        "n": 3, "m": 3,
        "ops": [[2, -1], [3, -2], [3, 1]],
        "expected": [5, 3, 1]
    },
    {
        "name": "極端單顆連續四次向前翻轉回原位",
        "n": 1, "m": 4,
        "ops": [[1, -1], [1, -1], [1, -1], [1, -1]],
        "expected": [1]
    },
    {
        "name": "兩顆骰子交替交換復原測試",
        "n": 2, "m": 3,
        "ops": [[1, -1], [1, 2], [1, 2]],
        "expected": [3, 1]
    }
]

print("=== f580. 骰子 全自動化測試報告 ===")
all_passed = True
for tc in test_cases:
    actual = solve_dice_simulation(tc["n"], tc["m"], tc["ops"])
    passed = (actual == tc["expected"])
    status = "✅ PASS" if passed else "❌ FAIL"
    if not passed:
        all_passed = False
    print(f"{status} | {tc['name']} ➔ 預期: {tc['expected']}, 實際: {actual}")

if all_passed:
    print("\n🎉 恭喜！所有官方與邊界壓力測試案例 100% 全數通過！可安心提交至 APCS 考場與 ZeroJudge！")
else:
    print("\n⚠️ 有測試資料未通過，請檢查四面輪替公式或對調邏輯！")
